In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 18
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 18
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
!bash start_data_swarm.sh

In [ ]:
!bash stop_data_swarm.sh

In [ ]:
!python -m _tools.check_and_heal_data

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [ ]:
#Запуск нескольких процессов в параллели
!bash start_swarm_amd.sh \
    --dataset_dir "data/processed/2000_2026_1d_18_3" \
    --bonus_ratio 0.2 \
    --min_delta 0.001 \
    --runs 500 --epochs 100 \
    --lr 2e-3 --l2_reg 1e-5 \
    --start_fold "fold_2010" \
    --factor 0.5 --patience 3 \
    --vram 10000/1 --stagger 0 \
    --keep 5 \
    --append \
    --arch attention
    #--arch cnn, conv1d+gru, mlp, attention
    #--track_trajectory
    #--init_pca_coord -148.0 -116.0 --init_pca_radius 20.0 \

In [ ]:
!./stop_swarm.sh
!python -m _tools.clean_lstm_models --keep 5

In [ ]:
!python run_all_ensembles.py --dataset_dir "data/processed/2000_2026_1d_18_3" --max_k 20

In [ ]:
%run _tools/plot_landscape.py --fold data/processed/2000_2026_1d_30_5/fold_2014

In [ ]:
%matplotlib inline
%run _tools/analyze_runs.py data/processed/2000_2026_1d_3_1/fold_2026 --runs 100 --arch attention

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.check_env_data

In [ ]:
!python -m _tools.train_rllib_pbt --population 6 --force --cpu --iterations 2500

In [ ]:
%load_ext tensorboard
%tensorboard --logdir data/processed/2000_2026_1d/rl_env/ray_results
#%tensorboard --logdir "C:\Users\Restorator\Documents\trader_test\trader_test\data\processed\2000_2026_1d\rl_env\ray_results"

In [18]:
!python -m _tools.export_to_git

🔍 Поиск лучшего чекпоинта через прямой разбор метаданных Ray Tune...

🏆 НАЙДЕН АБСОЛЮТНЫЙ ЧЕМПИОН ПОПУЛЯЦИИ!
  Trial ID: PPO_TradingEnv-v0_e7bfc_00003_3_2026-06-15_12-45-12
  Checkpoint: checkpoint_000005
  Средняя альфа-доходность (Return Mean): -27.42 б.п.
✅ Чекпоинт скопирован в: /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/champions/best_model
✅ Файл training_progress.csv успешно перенесен в champions.
✅ Файл evaluation_report.html успешно перенесен в champions.

🎉 Автономный экспорт успешно завершен!


In [20]:
!python -m _tools.evaluate_agent --checkpoint "/home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/champions/best_model"

I0000 00:00:1781529345.730994  240962 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781529345.732840  240962 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781529347.045314  240962 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781529347.046711  240962 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
🔄 Восстановление агента из /home/restorator/trader_test/data/processed/2000_2026_1d/rl_env/champions/best_model/policies/default_policy...
/home/restorator/trader_test/.venv-tf/lib/python3.1